# 15 — SQLite, SQL, and Transactions

SQLite is a lightweight embedded SQL database included with Python. Practice schema design, parameterized queries, CRUD, constraints, and transactions without creating persistent files.

*Intermediate · 75–120 min*

## You will learn

- Create tables with constraints and stable keys
- Insert, query, update, and delete records
- Use parameterized SQL to avoid injection
- Understand transaction commit, rollback, and connection cleanup

## Examples

### 1. Connecting and creating tables

sqlite3.connect(':memory:') makes an in-memory database that disappears when the connection is closed. CREATE TABLE defines columns and constraints. Always parameterize external values rather than interpolating them into SQL text.

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
try:
    connection.execute("""
        CREATE TABLE students (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL UNIQUE,
            score REAL NOT NULL CHECK(score BETWEEN 0 AND 100)
        )
    """)
    connection.executemany(
        "INSERT INTO students (name, score) VALUES (?, ?)",
        [("Ada", 91), ("Ben", 78), ("Cam", 85)]
    )
    print(connection.execute("SELECT name, score FROM students ORDER BY score DESC").fetchall())
finally:
    connection.close()

**Think about it:** Why does the CHECK constraint protect against an invalid score?

### 2. CRUD with bound parameters

CRUD means Create, Read, Update, Delete. Placeholder parameters keep query structure separate from data and prevent values from being interpreted as SQL syntax. SELECT can include filters, ordering, and limits.

In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as db:
    db.execute("CREATE TABLE products (sku TEXT PRIMARY KEY, price REAL)")
    db.executemany("INSERT INTO products VALUES (?, ?)", [("P1", 10), ("P2", 20)])
    db.execute("UPDATE products SET price = ? WHERE sku = ?", (12.5, "P1"))
    print(db.execute("SELECT * FROM products WHERE price >= ? ORDER BY sku", (10,)).fetchall())
    db.execute("DELETE FROM products WHERE sku = ?", ("P2",))
    print(db.execute("SELECT COUNT(*) FROM products").fetchone()[0])

**Think about it:** What is the difference between a value parameter and a table identifier in SQL?

### 3. Transactions and rollback

A transaction groups changes into an atomic unit. Successful operations can commit together; an error should roll back the entire unit when partial changes would be harmful. The connection context manager manages commit/rollback but does not itself close the connection.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER CHECK(balance >= 0))")
    db.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 40)])
    db.commit()
    try:
        with db:
            db.execute("UPDATE accounts SET balance = balance - 200 WHERE id = 1")
    except sqlite3.IntegrityError:
        print("Transfer rejected; transaction rolled back")
    print(db.execute("SELECT * FROM accounts ORDER BY id").fetchall())
finally:
    db.close()

**Think about it:** Why would a two-step transfer require both changes in the same transaction?

### 4. Aggregations and joins

SQL databases can summarize data using COUNT, SUM, AVG, GROUP BY, and join related tables. A foreign key relates records from different tables. In SQLite, enable foreign-key enforcement per connection when you rely on it.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("PRAGMA foreign_keys = ON")
    db.execute("CREATE TABLE categories (id INTEGER PRIMARY KEY, name TEXT)")
    db.execute("CREATE TABLE sales (category_id INTEGER REFERENCES categories(id), amount REAL)")
    db.executemany("INSERT INTO categories VALUES (?, ?)", [(1, "books"), (2, "pens")])
    db.executemany("INSERT INTO sales VALUES (?, ?)", [(1, 10), (1, 15), (2, 3)])
    query = """SELECT c.name, COUNT(*), SUM(s.amount)
               FROM sales s JOIN categories c ON s.category_id = c.id
               GROUP BY c.id, c.name ORDER BY c.name"""
    print(db.execute(query).fetchall())
finally:
    db.close()

**Think about it:** What happens when a sale references a category id that does not exist with foreign keys enabled?

## Exercises

**Exercise 1.** Design an SQLite table for tasks with a unique id and done flag.

In [ ]:
# Exercise 1
# Implement your solution here, and add tests.

**Exercise 2.** Create a function that inserts a task and returns its generated id.

In [ ]:
# Exercise 2
# Implement your solution here, and add tests.

**Exercise 3.** Write parameterized queries to filter records by status.

In [ ]:
# Exercise 3
# Implement your solution here, and add tests.

**Exercise 4.** Create a transaction that rolls back when a constraint is violated.

In [ ]:
# Exercise 4
# Implement your solution here, and add tests.

## Common mistakes

- Never construct SQL using untrusted user text as query syntax.
- The sqlite3 connection context manager does not close the connection.
- Remember PRAGMA foreign_keys = ON if you depend on foreign-key checks.

**Next:** [16 — HTTP, JSON APIs, and Reliable Clients](16_http_apis_and_clients.ipynb)